In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_scatter_with_coastline(df, figsize=(8, 6), point_size=20, color="red"):
    """
    Plots a scatter plot of coordinates with coastlines.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing coordinates.
        figsize (tuple): Figure size.
        point_size (int): Size of scatter points.
        color (str): Color of scatter points.
    """
    # Possible column name mappings
    lat_candidates = ["lat", "latitude", "Latitude", "LAT", "y", "Y"]
    lon_candidates = ["lon", "longitude", "Longitude", "LON", "x", "X"]

    # Find actual column names
    lat_col = next((col for col in df.columns if col in lat_candidates), None)
    lon_col = next((col for col in df.columns if col in lon_candidates), None)

    if lat_col is None or lon_col is None:
        raise ValueError("Could not find latitude/longitude columns in DataFrame.")

    # Create plot
    fig, ax = plt.subplots(
        figsize=figsize, 
        subplot_kw={"projection": ccrs.PlateCarree()}
    )
    # ax.set_global()
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")

    # Scatter plot
    ax.scatter(
        df[lon_col],
        df[lat_col],
        s=point_size,
        c=color,
        transform=ccrs.PlateCarree(),
        zorder=5
    )

    plt.show()


In [ ]:
copper_filename="<DATA_ROOT>/CopperLithium/NWMexico/Data/pcu_deps_pros.csv"

copper_df=pd.read_csv(copper_filename)

In [ ]:
copper_df.columns

In [ ]:
copper_df['tonnage_mt'].unique()

In [ ]:
copper_df= copper_df.replace(-9999., np.nan)
copper_df['tonnage_mt'].hist()

In [ ]:
plot_scatter_with_coastline(copper_df,color=copper_df['tonnage_mt'])

In [ ]:
copper_df['country'].unique()

In [ ]:
### filter out US and canada

copper_df_uc=copper_df[(copper_df['country']=="United States") | (copper_df['country']=="Canada")].copy()
print(f"Total number of copper deposits: {len(copper_df_uc)}")
print(f"Total number with Tonnage info:{len(copper_df_uc.dropna(subset=['tonnage_mt']))}")
      
plot_scatter_with_coastline(copper_df_uc,color=copper_df_uc['tonnage_mt'])

In [ ]:
copper_df_uc['age_ma'].hist()

In [ ]:
training_data_file="<HOME>/Downloads/ApplicationKalpa/13777155/stellar-data-mining-1.2/prepared_data/training_data_fz_seamount_LIP.csv"
training_df=pd.read_csv(training_data_file)

In [ ]:
training_df.head()

In [ ]:
training_df['label'].unique()

In [ ]:
training_df_positive=training_df[training_df['label']=='positive'].copy()
training_df_negative=training_df[(training_df['label']=='unlabelled') | (training_df['label']=='negative')].copy()

In [ ]:
training_gdf_positive=gpd.GeoDataFrame(training_df_positive,geometry=gpd.points_from_xy(training_df_positive['present_lon'],training_df_positive['present_lat']))

In [ ]:
copper_gdf=gpd.GeoDataFrame(copper_df,geometry=gpd.points_from_xy(copper_df['longitude'],copper_df['latitude']))

In [ ]:
training_postive_tonnage=gpd.sjoin_nearest(training_gdf_positive,copper_gdf[['tonnage_mt','geometry']],distance_col='distance')

In [ ]:
training_postive_tonnage['distance'].hist(bins=150)
plt.xlim([0,0.5])

In [ ]:
training_postive_tonnage['tonnage_mt'] = np.where(
    training_postive_tonnage['distance'] > 0.1,
    np.nan,
    training_postive_tonnage['tonnage_mt']
)
training_positive_tonnage=pd.DataFrame(training_postive_tonnage.drop(columns=['geometry']))
training_df_negative['tonnage_mt']=np.nan

training_df=pd.concat([training_positive_tonnage,training_df_negative])

In [ ]:
training_df_negative

In [ ]:
def compute_weights(df, tonnage_col="tonnage_mt"):
    weights = np.ones(len(df))  # default weight = 1
    
    if tonnage_col in df.columns:
        tonnage = df[tonnage_col].copy()
        
        # Replace -999, -9999 etc. with NaN
        tonnage = tonnage.replace([-999, -9999], np.nan)
        
        # Log scaling for known tonnage
        tonnage_weights = 1+np.log1p(tonnage)
        
        # Normalize to mean ~1
        # tonnage_weights = tonnage_weights / np.nanmean(tonnage_weights)
        
        # Fill missing tonnage with 1 (or median)
        tonnage_weights = tonnage_weights.fillna(1.0)
        # tonnage_weights = tonnage_weights / np.nanmean(tonnage_weights)
        
        weights = tonnage_weights.values
    
    return weights

In [ ]:
training_df["weights"]=compute_weights(training_df)

In [ ]:
training_df["weights"].hist(bins=200)

In [ ]:
training_df['label'].unique()
training_df['label_binary'] = training_df['label'].apply(lambda x: 1 if x == 'positive' else 0)

In [ ]:
# Shuffle the DataFrame
shuffled_df = training_df.sample(frac=1, random_state=42).reset_index(drop=True)

# # Save to CSV
# shuffled_df.to_csv("training_shuffled.csv", index=False)


In [ ]:
training_df.to_csv("<DATA_ROOT>/CopperLithium/NWMexico/PUBaggingModel/training_data.csv",index=False)

In [ ]:
grid_data_filename="<DATA_ROOT>/CopperLithium/NWMexico/PUBaggingModel/grid_data_fz_seamount_LIP.csv"
grid_data=pd.read_csv(grid_data_filename)
grid_data_gdf=gpd.GeoDataFrame(grid_data,geometry=gpd.points_from_xy(grid_data['present_lon'],grid_data['present_lat']))
grid_data_gdf.to_file("<DATA_ROOT>/CopperLithium/NWMexico/PUBaggingModel/grid_data_fz_seamount_LIP.gpkg")

In [ ]:
## Sample data from spatiotemporal output

reconstruction_time=240

### netcdf grid

